In [0]:
# Define and retrieve parameters
dbutils.widgets.text("catalog_name", "lending_risk_dev") # Default value for manual testing
CATALOG = dbutils.widgets.get("catalog_name")

print(f"Running setup for Catalog: {CATALOG}")

In [0]:
# Create Reference Table for Loan Purposes in Silver Layer
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.ref_loan_purposes (
        -- Business Columns
        loan_purpose STRING NOT NULL COMMENT 'Standardized loan purpose key',
        is_active BOOLEAN NOT NULL COMMENT 'SCD-1 Soft Delete Flag: false for decommissioned',
        
        -- Silver Metadata (Audit Columns)
        _updated_at TIMESTAMP NOT NULL,
        _source_file STRING NOT NULL,
        _processed_by STRING NOT NULL, 
        _batch_id STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',  
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true',
        'delta.enableDeletionVectors' = 'true',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    );
    """
)

In [0]:
# Create Customers Table in Silver Layer
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.customers (
    member_id                 STRING NOT NULL,
    emp_title                 STRING,
    emp_length                INT,
    home_ownership            STRING,
    annual_income             FLOAT,
    address_state             STRING,
    address_zipcode           STRING,
    address_country           STRING,
    grade                     STRING,
    sub_grade                 STRING,
    verification_status       STRING,
    total_high_credit_limit   FLOAT,
    application_type          STRING,
    joint_annual_income       FLOAT,
    verification_status_joint STRING,
    _ingested_at              TIMESTAMP,
    _source_file              STRING,
    _bronze_version           LONG,
    _updated_at               TIMESTAMP,
    _batch_id                  STRING
)
USING DELTA
CLUSTER BY (member_id) 
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableDeletionVectors' = 'true',
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
"""
)

In [0]:
# Create Loans Table in Silver Layer
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.loans (
    loan_id               STRING NOT NULL,
    member_id             STRING NOT NULL,
    loan_amount           FLOAT,
    funded_amount         FLOAT,
    interest_rate         FLOAT,
    monthly_installment   FLOAT,
    issue_date            STRING,
    loan_status           STRING,
    title                 STRING,
    loan_term_years       INT,
    loan_purpose          STRING,
    
    -- Lineage & Metadata
    _ingested_at          TIMESTAMP,
    _source_file          STRING,
    _bronze_version       LONG,
    _updated_at           TIMESTAMP,
    _batch_id             STRING 
) 
USING DELTA
CLUSTER BY (member_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableDeletionVectors' = 'true',
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
"""
)

In [0]:
# Create Repayments Table in Silver Layer
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.repayments (
  loan_id                   STRING NOT NULL,
  total_principal_received  FLOAT,
  total_interest_received   FLOAT,
  total_late_fee_received   FLOAT,
  total_payment_received    FLOAT,
  last_payment_amount       FLOAT,
  last_payment_date         STRING,      -- Input data reasons
  next_payment_date         STRING,      -- Input data reasons
  
  -- Lineage & Metadata
  _ingested_at              TIMESTAMP,
  _source_file              STRING,
  _bronze_version           LONG,
  _updated_at               TIMESTAMP,
  _batch_id                 STRING      
) 
USING DELTA
CLUSTER BY (loan_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableDeletionVectors' = 'true',
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
"""
)

In [0]:
# Create Defaulters Delinq Table in Silver Layer
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.defaulters_delinq 
(
  member_id                 STRING NOT NULL,
  delinq_2yrs               INT,
  delinq_amnt               FLOAT,
  mths_since_last_delinq    INT,
  
  -- Lineage & Metadata
  _ingested_at              TIMESTAMP,
  _source_file              STRING,
  _bronze_version           LONG,
  _updated_at               TIMESTAMP,
  _batch_id                 STRING
) 
USING DELTA
CLUSTER BY (member_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableDeletionVectors' = 'true',
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
"""
)

In [0]:
# Create Defaulters Inquiry Table in Silver Layer
spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.defaulters_inquiry 
(
  member_id                 STRING NOT NULL,
  pub_rec                   INT,       -- NOT FLOAT
  pub_rec_bankruptcies      INT,       -- NOT FLOAT
  inq_last_6mths            INT,       -- NOT FLOAT
  
  -- Lineage & Metadata
  _ingested_at              TIMESTAMP,
  _source_file              STRING,
  _bronze_version           LONG,
  _updated_at               TIMESTAMP,
  _batch_id                 STRING     
) 
USING DELTA
CLUSTER BY (member_id) 
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.enableDeletionVectors' = 'true',
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
"""
)